<a href="https://colab.research.google.com/github/ViSaReVe/triton-serving-lab/blob/main/Deliverable1_Step2_3_triton_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deliverable 1 · Steps 2 & 3 — Serve on Triton + benchmark (Colab T4)

Your Mac is **arm64**, so the official Triton container would be a ~15 GB x86 image under emulation — slow, and any
benchmark from it would be meaningless. Instead we run **real Triton** here on Colab's x86 + T4 via **PyTriton**
(`pip install nvidia-pytriton` ships the actual Triton server binary — no Docker required).

**Step 2:** stand up Triton, send your first inference request.  
**Step 3:** benchmark — dynamic batching ON vs OFF, and GPU vs CPU — to fill your README table.

Runtime → Change runtime type → **T4 GPU**.


## 0. Setup
`onnxruntime-gpu`'s newest build targets CUDA 13, which Colab may not have (that's the `libcudart.so.13` error from
Step 1). We pin a CUDA-12 build and **fall back to CPU automatically**, then print which provider is actually active —
always verify this rather than assuming you're on GPU.


In [4]:
!nvidia-smi -L
# onnxruntime-gpu 1.19.2 = CUDA-12 build (Colab has CUDA 12, not 13 -> avoids the libcudart.so.13 error).
# numpy pinned to 1.26.4 to match PyTriton.
!pip -q install nvidia-pytriton 'onnxruntime-gpu==1.19.2' 'numpy==1.26.4' 'tritonclient[http]'
print('\n>>> INSTALL DONE. Now: Runtime -> Restart session, then run from cell 0c. Do NOT re-run this cell. <<<')


GPU 0: Tesla T4 (UUID: GPU-867c35f1-77a3-85e7-aac2-23c057269510)

>>> INSTALL DONE. Now: Runtime -> Restart session, then run from cell 0c. Do NOT re-run this cell. <<<


## 1. Get the model
Upload `audio_cnn.onnx` from Step 1. (Once your repo is pushed you can instead clone it — see the commented line.)


In [1]:
import numpy as np, onnxruntime as ort
print('numpy', np.__version__, '(should be 1.26.4)')
print('onnxruntime', ort.__version__)
print('available providers:', ort.get_available_providers())


numpy 1.26.4 (should be 1.26.4)
onnxruntime 1.19.2
available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [2]:
import os
ONNX = 'audio_cnn.onnx'
if not os.path.exists(ONNX):
    from google.colab import files
    up = files.upload()
    ONNX = [f for f in up if f.endswith('.onnx')][0]
# alt once pushed:
# !git clone https://github.com/ViSaReVe/triton-serving-lab.git
# ONNX = 'triton-serving-lab/model_repository/audio_cnn/1/model.onnx'
print('model:', ONNX)


model: audio_cnn.onnx


## 2. Load the ONNX on the GPU
We create the inference session Triton will call. Note we **check the provider actually in use** — if CUDA didn't
load, you'd silently benchmark CPU and think it was GPU.


In [3]:
prov = ['CUDAExecutionProvider','CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
sess_gpu = ort.InferenceSession(ONNX, providers=prov)
sess_cpu = ort.InferenceSession(ONNX, providers=['CPUExecutionProvider'])
ACTIVE = sess_gpu.get_providers()[0]
print('ACTIVE PROVIDER:', ACTIVE)   # want CUDAExecutionProvider

x = np.random.randn(4,1,128,128).astype('float32')
print('smoke test ->', sess_gpu.run(['logits'], {'input': x})[0].shape)   # (4,10)


ACTIVE PROVIDER: CUDAExecutionProvider
smoke test -> (4, 10)


## 3. STEP 2 — bind the model to Triton and start the server
This is real Triton. Two models, identical weights, differing only in the batcher:
- **audio_cnn** — `DynamicBatcher(preferred_batch_size=[4,8,16], max_queue_delay_microseconds=3000)`
- **audio_cnn_nobatch** — `max_batch_size=1`, no fusing (the control group)

Same knobs as your `config.pbtxt`, expressed in Python. `@batch` makes Triton hand your function a *fused batch*.
`triton.run()` is non-blocking so we can send requests from the same notebook.


In [4]:
from pytriton.decorators import batch
from pytriton.model_config import ModelConfig, Tensor, DynamicBatcher
from pytriton.triton import Triton

@batch
def infer(input):
    return {'logits': sess_gpu.run(['logits'], {'input': input})[0]}

triton = Triton()
triton.bind(model_name='audio_cnn', infer_func=infer,
    inputs=[Tensor(name='input', dtype=np.float32, shape=(1,128,128))],
    outputs=[Tensor(name='logits', dtype=np.float32, shape=(10,))],
    config=ModelConfig(max_batch_size=32,
        batcher=DynamicBatcher(preferred_batch_size=[4,8,16], max_queue_delay_microseconds=3000)))
triton.bind(model_name='audio_cnn_nobatch', infer_func=infer,
    inputs=[Tensor(name='input', dtype=np.float32, shape=(1,128,128))],
    outputs=[Tensor(name='logits', dtype=np.float32, shape=(10,))],
    config=ModelConfig(max_batch_size=1))

triton.run()      # non-blocking; HTTP :8000, gRPC :8001, metrics :8002
import time; time.sleep(6)
print('Triton is up.')


I0718 21:33:12.945824 4323 pinned_memory_manager.cc:277] "Pinned memory pool is created at '0x7d4b20000000' with size 268435456"
I0718 21:33:12.949112 4323 cuda_memory_manager.cc:107] "CUDA memory pool is created on device 0 with size 67108864"
I0718 21:33:12.954295 4323 server.cc:604] 
+------------------+------+
| Repository Agent | Path |
+------------------+------+
+------------------+------+

I0718 21:33:12.954336 4323 server.cc:631] 
+---------+------+--------+
| Backend | Path | Config |
+---------+------+--------+
+---------+------+--------+

I0718 21:33:12.954354 4323 server.cc:674] 
+-------+---------+--------+
| Model | Version | Status |
+-------+---------+--------+
+-------+---------+--------+

I0718 21:33:12.985234 4323 metrics.cc:877] "Collecting metrics for GPU 0: Tesla T4"
I0718 21:33:12.999779 4323 metrics.cc:770] "Collecting CPU metrics"
I0718 21:33:12.999927 4323 tritonserver.cc:2598] 
+----------------------------------+------------------------------------------+
|

### Your first inference request
The Step-2 milestone: your trained EE541 model answering over HTTP through Triton.


In [5]:
import tritonclient.http as httpclient
client = httpclient.InferenceServerClient(url='localhost:8000')
print('server live:', client.is_server_live())
for m in ('audio_cnn','audio_cnn_nobatch'):
    print(f'  {m} ready:', client.is_model_ready(m))

x1 = np.random.randn(1,1,128,128).astype('float32')
inp = httpclient.InferInput('input', x1.shape, 'FP32'); inp.set_data_from_numpy(x1)
r = client.infer('audio_cnn', inputs=[inp], outputs=[httpclient.InferRequestedOutput('logits')])
logits = r.as_numpy('logits')
print('logits:', logits.shape, '| predicted class:', int(logits.argmax()))


server live: True
  audio_cnn ready: True
  audio_cnn_nobatch ready: True
logits: (1, 10) | predicted class: 9


## 4. STEP 3 — benchmark
Fire many concurrent **single-sample** requests and time each. That's the realistic serving pattern: many independent
clients. Dynamic batching wins by *fusing* those into one forward pass — fewer, larger passes use the GPU far better.


In [8]:
import time
import tritonclient.grpc as grpcclient
from concurrent.futures import ThreadPoolExecutor, as_completed

def pct(v, p):
    s = sorted(v); k = max(0, min(len(s)-1, int(round((p/100)*(len(s)-1))))); return s[k]

def bench(model, n=400, conc=16):
    cl = grpcclient.InferenceServerClient(url='localhost:8001')   # gRPC, not HTTP
    xx = np.random.randn(1, 1, 128, 128).astype('float32')
    def one():
        i = grpcclient.InferInput('input', xx.shape, 'FP32'); i.set_data_from_numpy(xx)
        t = time.perf_counter()
        cl.infer(model, inputs=[i], outputs=[grpcclient.InferRequestedOutput('logits')])
        return (time.perf_counter() - t) * 1000
    one()  # warm-up
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=conc) as ex:
        lat = [f.result() for f in as_completed([ex.submit(one) for _ in range(n)])]
    wall = time.perf_counter() - t0
    return {'model': model, 'conc': conc, 'rps': round(n/wall, 1),
            'p50': round(pct(lat,50),2), 'p95': round(pct(lat,95),2), 'p99': round(pct(lat,99),2)}

rows = []
for conc in (1, 8, 16, 32):
    for m in ('audio_cnn', 'audio_cnn_nobatch'):
        r = bench(m, 400, conc); rows.append(r); print(r)

{'model': 'audio_cnn', 'conc': 1, 'rps': 128.1, 'p50': 6.76, 'p95': 9.16, 'p99': 17.59}
{'model': 'audio_cnn_nobatch', 'conc': 1, 'rps': 249.5, 'p50': 3.75, 'p95': 4.89, 'p99': 7.69}
{'model': 'audio_cnn', 'conc': 8, 'rps': 559.7, 'p50': 11.09, 'p95': 23.73, 'p99': 85.96}
{'model': 'audio_cnn_nobatch', 'conc': 8, 'rps': 282.5, 'p50': 27.26, 'p95': 31.66, 'p99': 39.51}
{'model': 'audio_cnn', 'conc': 16, 'rps': 505.2, 'p50': 21.06, 'p95': 68.26, 'p99': 180.33}
{'model': 'audio_cnn_nobatch', 'conc': 16, 'rps': 229.8, 'p50': 67.73, 'p95': 78.45, 'p99': 84.59}
{'model': 'audio_cnn', 'conc': 32, 'rps': 367.5, 'p50': 79.49, 'p95': 190.86, 'p99': 203.39}
{'model': 'audio_cnn_nobatch', 'conc': 32, 'rps': 283.5, 'p50': 109.12, 'p95': 120.62, 'p99': 121.32}


### Results table (paste into your README)


In [9]:
print(f'ACTIVE PROVIDER: {ACTIVE}\n')
print('| model | concurrency | throughput (req/s) | p50 ms | p95 ms | p99 ms |')
print('|---|---|---|---|---|---|')
for r in rows:
    name = 'dynamic batching' if r['model']=='audio_cnn' else 'no batching'
    print(f"| {name} | {r['conc']} | {r['rps']} | {r['p50']} | {r['p95']} | {r['p99']} |")


ACTIVE PROVIDER: CUDAExecutionProvider

| model | concurrency | throughput (req/s) | p50 ms | p95 ms | p99 ms |
|---|---|---|---|---|---|
| dynamic batching | 1 | 128.1 | 6.76 | 9.16 | 17.59 |
| no batching | 1 | 249.5 | 3.75 | 4.89 | 7.69 |
| dynamic batching | 8 | 559.7 | 11.09 | 23.73 | 85.96 |
| no batching | 8 | 282.5 | 27.26 | 31.66 | 39.51 |
| dynamic batching | 16 | 505.2 | 21.06 | 68.26 | 180.33 |
| no batching | 16 | 229.8 | 67.73 | 78.45 | 84.59 |
| dynamic batching | 32 | 367.5 | 79.49 | 190.86 | 203.39 |
| no batching | 32 | 283.5 | 109.12 | 120.62 | 121.32 |


### GPU vs CPU (same model, same ONNX)
Direct engine-level comparison, no serving overhead — isolates the hardware effect.


In [10]:
def raw(sess, bs=8, iters=100):
    xx=np.random.randn(bs,1,128,128).astype('float32'); sess.run(['logits'],{'input':xx})
    t0=time.perf_counter()
    for _ in range(iters): sess.run(['logits'],{'input':xx})
    ms=(time.perf_counter()-t0)/iters*1000; return round(ms,2), round(bs/(ms/1000),1)
for label,s in ((f'GPU {ACTIVE}',sess_gpu),('CPU',sess_cpu)):
    ms,rps=raw(s); print(f'{label:34s} batch=8: {ms:7.2f} ms  ~{rps:9.1f} samples/s')


GPU CUDAExecutionProvider          batch=8:    1.45 ms  ~   5500.1 samples/s
CPU                                batch=8:   24.54 ms  ~    325.9 samples/s


## 5. Shut down + what you learned


In [11]:
triton.stop()
print('Triton stopped.')


Signal (2) received.
I0718 21:36:47.953077 4323 http_server.cc:248] "Timeout 30: Found 2 HTTP service connections"
I0718 21:36:48.953214 4323 http_server.cc:248] "Timeout 29: Found 2 HTTP service connections"
I0718 21:36:49.953427 4323 http_server.cc:248] "Timeout 28: Found 2 HTTP service connections"
I0718 21:36:50.953584 4323 http_server.cc:248] "Timeout 27: Found 2 HTTP service connections"
I0718 21:36:51.953842 4323 http_server.cc:248] "Timeout 26: Found 2 HTTP service connections"
I0718 21:36:52.954006 4323 http_server.cc:248] "Timeout 25: Found 2 HTTP service connections"
I0718 21:36:53.954175 4323 http_server.cc:248] "Timeout 24: Found 2 HTTP service connections"
I0718 21:36:54.954332 4323 http_server.cc:248] "Timeout 23: Found 2 HTTP service connections"
I0718 21:36:55.954482 4323 http_server.cc:248] "Timeout 22: Found 2 HTTP service connections"
I0718 21:36:56.954628 4323 http_server.cc:248] "Timeout 21: Found 2 HTTP service connections"
I0718 21:36:57.954771 4323 http_server.

I0718 21:37:18.374195 4467 pb_stub.cc:2145]  Non-graceful termination detected. 
/usr/lib/python3.12/multiprocessing/resource_tracker.py:279: UserWarning: resource_tracker: There appear to be 1 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
I0718 21:37:18.428013 4384 pb_stub.cc:2145]  Non-graceful termination detected. 
/usr/lib/python3.12/multiprocessing/resource_tracker.py:279: UserWarning: resource_tracker: There appear to be 24 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
Triton stopped.


**What Step 2–3 proved:**
- You served a *real trained model* through **real Triton** and got answers over HTTP.
- Dynamic batching is a **latency↔throughput dial**: it deliberately waits up to `max_queue_delay_microseconds` to
  fuse requests. At concurrency 1 it can't help (nothing to fuse); as concurrency rises the throughput gap widens.
- You verified the **active execution provider** instead of assuming GPU.

**Self-check:** (1) Why does batching barely help at concurrency=1? (2) If your SLO were 'p99 < 10 ms', which way do
you turn `max_queue_delay`? (3) This CNN is compute-bound — why does that make batching effective, and why is LLM
*decode* different? (→ leads into Dynamo, Deliverable 2.)

**Next:** paste the results table into your README, commit (`File → Save a copy in GitHub`), and the resume line is true.
